# NQCT SDK walkthrough — submit OpenQASM

End-to-end notebook for the Phase 1.5 **direct QASM** flow against a live NQCT Cloud stack:

1. Authenticate and discover backends
2. Submit inline OpenQASM 3 via `client.submit_job()` → `POST /jobs`
3. Poll the async **Job** until it finishes
4. Fetch results (and logs on failure)

**Prerequisites**

1. Point at **production** (`https://api.nqct.org/api/v1`) or start local `nqct-cloud` with `nqct start`
2. Create an API key in the web UI profile page
3. Link at least one **online simulator** backend in the admin UI
4. Install notebook deps: `uv sync --extra notebook`

**Credentials in notebooks (read this)**

A Jupyter kernel is a **separate process**. Environment variables you `export` in a terminal are **not** always visible to the notebook kernel (especially when the IDE launches Jupyter separately).

Use **one** of these:

| Method | When to use |
|--------|-------------|
| **Set `NOTEBOOK_API_KEY` in the next cell** | Easiest when the kernel does not inherit your shell env |
| **`NQCTClient.save_account(...)`** | Persist to `~/.nqct/credentials.json` |
| **Export + launch Jupyter from that shell** | `uv run jupyter lab examples/sdk_walkthrough.ipynb` |

Never commit API keys.

In [ ]:
from __future__ import annotations

import os
from pprint import pprint

from nqct import Client, NQCTClient
from nqct.auth.credentials import load_profile
from nqct.exceptions import JobFailedError, JobNotCompleteError, JobTimeoutError, NQCTError

# Option A: set your key here if the kernel does not inherit shell env
# NOTEBOOK_API_KEY = "nqct_your_key_here"
# NOTEBOOK_URL = "https://api.nqct.org/api/v1"  # production
# NOTEBOOK_URL = "http://localhost:8000/api/v1"  # local
NOTEBOOK_API_KEY: str | None = "YOUR_API_KEY_HERE"
NOTEBOOK_URL: str | None = None

# Option B: shell env (works when kernel inherited it, e.g. uv run jupyter lab)
NQCT_URL = NOTEBOOK_URL or os.environ.get("NQCT_URL") or "https://api.nqct.org/api/v1"
NQCT_API_KEY = NOTEBOOK_API_KEY or os.environ.get("NQCT_API_KEY")

# Option C: ~/.nqct/credentials.json from save_account()
if not NQCT_API_KEY:
    try:
        profile = load_profile()
        NQCT_URL = profile.get("url", NQCT_URL)
        NQCT_API_KEY = profile.get("api_key")
        print("Loaded credentials from ~/.nqct/credentials.json")
    except (FileNotFoundError, KeyError):
        pass

if not NQCT_API_KEY:
    raise RuntimeError(
        "No API key found by the notebook kernel.\n\n"
        "Shell exports may not reach the notebook kernel.\n"
        "Fix: set NOTEBOOK_API_KEY in this cell, or run save_account() below,\n"
        "or start Jupyter from the shell where you exported:\n"
        "  uv run jupyter lab examples/sdk_walkthrough.ipynb"
    )

print(f"API URL: {NQCT_URL}")
print(f"API key set: yes")
print(f"Python: {__import__('sys').executable}")

## Optional: save credentials to disk

Uncomment to persist `~/.nqct/credentials.json` (mode `0600`). After that you can use `NQCTClient()` with no arguments.

In [ ]:
# NQCTClient.save_account(url=NQCT_URL, api_key=NQCT_API_KEY)
# client = NQCTClient()  # loads ~/.nqct or env

client = NQCTClient(url=NQCT_URL, api_key=NQCT_API_KEY)
assert Client is NQCTClient
client

## Authentication — `GET /auth/me`

In [ ]:
profile = client.me()
pprint({k: profile.get(k) for k in ("id", "username", "email", "full_name", "is_superuser")})

## Backends — `GET /backends`

List catalog backends, inspect one online simulator, and pick the least-busy target for the QASM job below.

In [ ]:
backends = client.backends(limit=20)
print(f"Found {len(backends)} backend(s)")
for b in backends:
    print(f"  {b.id:30} {b.name:25} {b.type:10} {b.status}")

In [ ]:
online = client.backends(status="online", type="simulator")
if not online:
    print("No online simulators — link a backend in the NQCT admin UI.")
else:
    backend = online[0]
    detail = client.backend(backend.id)
    print(f"Detail: {detail.name} ({detail.id}), qubits={detail.qubits}")

    queue = detail.queue_status()
    print(
        f"Queue depth={queue.queue_depth}, "
        f"queued={queue.queued}, running={queue.running}, "
        f"est. wait={queue.estimated_wait_time_seconds:.1f}s"
    )

In [ ]:
try:
    picked = client.least_busy(type="simulator")
    print(f"Least busy simulator: {picked.name} ({picked.id})")
except LookupError as exc:
    print(exc)

## Submit OpenQASM — `POST /jobs` (end-to-end)

This is the main SDK path for running a circuit on a **managed** backend:

1. Build (or paste) an OpenQASM 3 program
2. Call `client.submit_job(...)` → platform creates an async **Job**
3. Poll with `job.wait()` until `done` / `failed` / `cancelled`
4. Read `job.result()` when `done`

Use `source="api"` so Jobs UI filters can distinguish SDK submits from UI `direct_qasm` jobs.

Set `BACKEND_ID` to pin a backend, or leave `None` to use `least_busy(type="simulator")`.
Set `RUN_SUBMIT = False` to skip the live call.

In [ ]:
# --- 1) OpenQASM 3 program (Bell state) ---------------------------------
BELL_QASM = """OPENQASM 3.0;
include "stdgates.inc";
qubit[2] q;
bit[2] c;
h q[0];
cx q[0], q[1];
c = measure q;
"""

# --- 2) Submit options --------------------------------------------------
BACKEND_ID: str | None = None  # e.g. "qiskit-aer-local"; None → least_busy simulator
SHOTS = 1024
OPTIMIZATION_LEVEL = 1  # 0–3; folded into execution_config.simulator
RUN_SUBMIT = True
SUBMIT_TIMEOUT = 600  # seconds for job.wait()
POLL_INTERVAL = 5

if not RUN_SUBMIT:
    print("Skipped submit (RUN_SUBMIT=False).")
else:
    # --- 3) Resolve backend ---------------------------------------------
    try:
        if BACKEND_ID:
            backend = client.backend(BACKEND_ID)
        else:
            backend = client.least_busy(type="simulator")
    except (LookupError, NQCTError) as exc:
        print(f"Skipped submit — no backend: {exc}")
    else:
        print(f"Backend: {backend.name} ({backend.id}) type={backend.type} status={backend.status}")
        print("Submitting OpenQASM 3 via POST /jobs ...")
        print(BELL_QASM)

        # --- 4) Create job ----------------------------------------------
        job = client.submit_job(
            qasm=BELL_QASM,
            backend_id=backend.id,
            shots=SHOTS,
            source="api",
            optimization_level=OPTIMIZATION_LEVEL,
            metadata={
                "label": "sdk-walkthrough-bell",
                "example": "sdk_walkthrough.ipynb",
            },
        )
        print(
            f"Queued job {job.id}\n"
            f"  status={job.status}  source={job.source}  shots={job.shots}\n"
            f"  backend={job.backend_id}  queue_position={job.queue_position}\n"
            f"  booking_bypass={job.booking_bypass}"
        )

        # --- 5) Poll until terminal -------------------------------------
        try:
            finished = job.wait(timeout=SUBMIT_TIMEOUT, interval=POLL_INTERVAL)
            print(f"Finished: status={finished.status}")
        except JobTimeoutError as exc:
            print(f"Timed out waiting for job {job.id}: {exc.message}")
            finished = job.refresh()
        except JobFailedError as exc:
            print(f"Job {job.id} failed: {exc.message}")
            finished = job.refresh()

        # --- 6) Results or diagnostics ----------------------------------
        print(f"Final status={finished.status}  execution_time={finished.execution_time_seconds}")
        if finished.status == "done":
            results = finished.result()
            print("Results:")
            pprint(results)
            counts = results.get("counts") if isinstance(results, dict) else None
            if counts:
                print("Counts:")
                for bitstring, n in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
                    print(f"  {bitstring}: {n}")
        else:
            if finished.error_message:
                print(f"error_message: {finished.error_message}")
            print("Logs:")
            try:
                for line in finished.logs()[:30]:
                    print(f"  {line}")
            except NQCTError as exc:
                print(f"  (could not fetch logs: {exc.message})")

## Inspect the submitted job — `GET /jobs/{id}`

Re-fetch the last submit (if you ran the cell above) or fall back to the most recent `source="api"` job.

In [ ]:
inspect_id = None
if "finished" in dir() and finished is not None:
    inspect_id = finished.id
elif "job" in dir() and job is not None:
    inspect_id = job.id
else:
    recent_api = client.jobs(source="api", limit=1)
    if recent_api:
        inspect_id = recent_api[0].id

if inspect_id is None:
    print("No job to inspect — run the submit cell first.")
else:
    inspected = client.job(inspect_id)
    print(
        f"Job {inspected.id}\n"
        f"  status={inspected.status}  source={inspected.source}\n"
        f"  backend={inspected.backend_id}  shots={inspected.shots}\n"
        f"  submit={inspected.submit_time}  start={inspected.start_time}  end={inspected.end_time}"
    )
    try:
        logs = inspected.logs()
        print("Logs (first 15):")
        for line in logs[:15]:
            print(f"  {line}")
    except NQCTError as exc:
        print(f"Could not fetch logs: {exc.message}")

    if inspected.status == "done":
        pprint(inspected.result())
    else:
        try:
            inspected.result()
        except JobNotCompleteError as exc:
            print(f"Expected while not done: {exc.message}")

## List jobs — `GET /jobs`

Filter by `source="api"` to see SDK submits from this walkthrough.

In [ ]:
all_jobs = client.jobs(limit=10)
print(f"Recent jobs (any source): {len(all_jobs)}")
for j in all_jobs:
    print(
        f"  {j.id}  source={j.source or '-':15} {j.status:10} "
        f"shots={j.shots} backend={j.backend_id} queue={j.queue_position}"
    )

api_jobs = client.jobs(source="api", limit=5)
print(f"\nRecent API-source jobs: {len(api_jobs)}")
for j in api_jobs:
    print(f"  {j.id}  {j.status:10} shots={j.shots} backend={j.backend_id}")

## Optional: custom `execution_config`

Simulator options can be passed as kwargs, or you can supply a full `execution_config` envelope
(when provided explicitly, other simulator/hardware kwargs are ignored).

Uncomment to run a second job with a fake backend noise model name (Aer-compatible backends only).

In [ ]:
RUN_ADVANCED = False  # set True to enqueue a second job

if not RUN_ADVANCED:
    print("Skipped advanced submit (RUN_ADVANCED=False).")
else:
    try:
        backend = client.backend(BACKEND_ID) if BACKEND_ID else client.least_busy(type="simulator")
    except (LookupError, NQCTError) as exc:
        print(f"Skipped advanced submit — no backend: {exc}")
    else:
        job2 = client.submit_job(
            qasm=BELL_QASM,
            backend_id=backend.id,
            shots=2048,
            source="api",
            optimization_level=2,
            # fake_backend_name="FakeManilaV2",  # optional; backend must support it
            metadata={"label": "sdk-walkthrough-bell-opt2"},
        )
        print(f"Advanced job {job2.id} queued at {job2.queue_position}")
        done2 = job2.wait(timeout=SUBMIT_TIMEOUT, interval=POLL_INTERVAL)
        print(f"Finished: {done2.status}")
        if done2.status == "done":
            pprint(done2.result())

## Cleanup

In [ ]:
client.close()
print("Client closed.")